# 10 · Retrain & evaluate the Type-2 (sentence) HalluShift head

The deployed sentence HalluShift head (`hal_det_sentence_s1`) **collapses on transfer** (cross-eval squad/triviaqa flag *everything* — confusion matrices with TN=0/FN=0, AUROC 0.45–0.55). Proven cause (two compounding training defects), not a dead signal —

**A. A `StandardScaler` landmine.** In the sentence regime ~45/71 HalluShift divergence/cosine features are near-constant (std≈0), so the scaler stored `scale_` as low as **1.6e-13**. At inference the moment those features deviate from the training mean by even 1e-4 (any new dataset), `(x-mean)/1.6e-13` explodes to ~1e8–1e12 and saturates the LayerNorm `CombinedNN` → constant output → flags everything.

**B. Underfit MLP.** In-distribution held-out the deployed `CombinedNN` reaches only ~0.65 (flat) while a plain logistic regression on the *same features and split* hits ~0.79.

This notebook: **(0)** optionally **builds a larger / more-diverse training set** (GPU), **(1)** loads the (judge-labelled) build + one shared split, **(2)** reproduces the landmine proof on the deployed head, **(3)** fixes the scaler, **(4)** retrains a `CombinedNN` + logreg, **(5)** full held-out eval, **(6)** perturbation stability, **(7)** real cross-dataset transfer eval (GPU), **(8)** gated save. Run in se_probes_env. **Save is OFF by default — review first.**

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
sys.path.insert(0, os.path.abspath(os.path.join('..', 'tools')))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch, pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import metrics as M
from classifier import CombinedNN
ROOT = os.path.abspath('..')
DATA = os.path.join(ROOT, 'data'); ART = os.path.join(ROOT, 'artifacts', 'hallushift')
HS = [f'hs_feat_{j:02d}' for j in range(71)]
SEED = 42
print('torch', torch.__version__)

## 0 · Config — choose the training set (and whether to scale it up)
- **Reproduce the baseline:** leave everything as-is (`TRAIN_TAG='s1j'`, `BUILD=False`) — trains on the cached TriviaQA-only judge-labelled build (~500 rows).
- **Scale up (recommended next step):** set `BUILD=True` and pick a new `TRAIN_TAG` (e.g. `'s2'`). Transfer is helped more by **dataset diversity** than raw volume, so `BUILD_DATASETS` defaults to triviaqa+squad. `BUILD_OFFSETS` keeps the squad *training* questions **disjoint** from the squad *transfer* set (offset 0) so there is no leakage. This build is a **GPU** pass (one 8B load, a few min).

In [ ]:
# ---- training set ------------------------------------------------------------------------
TRAIN_TAG = 's1j'          # cached build to train on: 's1j' (TriviaQA-only baseline) or a NEW tag you build below
BUILD     = False          # True -> regenerate a larger/multi-dataset training set (GPU). False -> reuse cache.
BUILD_DATASETS = ['triviaqa', 'squad']     # diversity helps transfer more than raw volume
BUILD_N        = 1200                       # questions PER dataset (refusal-drop yields fewer rows)
BUILD_OFFSETS  = {'triviaqa': 0, 'squad': 500}  # squad train offset disjoint from the squad TRANSFER set (offset 0)
# ---- transfer eval target (held-out) -----------------------------------------------------
TRANSFER_DS, TRANSFER_N, TRANSFER_OFFSET = 'squad', 200, 0   # squad offset 0 is already cached from the audit
# ------------------------------------------------------------------------------------------
print(f'TRAIN_TAG={TRAIN_TAG} | BUILD={BUILD} | transfer={TRANSFER_DS}@{TRANSFER_OFFSET}')

### 0b · (optional, GPU) build the larger training set
Generates one factual sentence per question for each dataset (sentence regime, `DEMO_FACTUAL` prompt), caches the 71-d HalluShift features + **judge** labels, drops refusals → `data/claims_sent_<TRAIN_TAG>.parquet`. Skipped when `BUILD=False`.

In [ ]:
build_path = os.path.join(DATA, f'claims_sent_{TRAIN_TAG}.parquet')
if BUILD:
    import retrain   # per-dataset offsets (train_claim_heads.build only takes one offset for all datasets)
    parts = []
    for ds in BUILD_DATASETS:
        off = BUILD_OFFSETS.get(ds, 0)
        di, _ = retrain.gen_and_cache(ds, n=BUILD_N, offset=off, regime='sentence',
                                      label_method='llm_judge', drop_refusals=True)
        di['source'] = f'qa:{ds}'; parts.append(di)
    bdf = pd.concat(parts, ignore_index=True)
    bdf.to_parquet(build_path)
    print('BUILT', build_path, bdf.shape, '|', bdf['source'].value_counts().to_dict(),
          f'| halluc={bdf["hallucination"].mean()*100:.1f}%')
else:
    print(f'BUILD=False -> using cached {os.path.relpath(build_path, ROOT)}')

## 1 · Load the chosen build + one shared split
`data/claims_sent_<TRAIN_TAG>.parquet` carries the cached 71-d HalluShift features and the `hallucination` (judge) labels. One stratified 75/25 split (seed 42) is reused by every model so the held-out AUROCs are directly comparable; a small validation slice is carved out of TRAIN for early stopping (TEST stays untouched).

In [ ]:
df = pd.read_parquet(build_path).reset_index(drop=True)
y = df['hallucination'].to_numpy().astype(int)
X = df[HS].to_numpy(np.float64)
src = df['source'].value_counts().to_dict() if 'source' in df else 'n/a'
print(f'n={len(df)} | halluc={y.mean()*100:.1f}% | sources={src}')
if 'hallucination_refmatch' in df:   # s1j only: how many substring labels the judge corrected
    print('judge corrected', int((df.hallucination.to_numpy() != df.hallucination_refmatch.to_numpy()).sum()),
          'substring labels')
tr_all, te = train_test_split(np.arange(len(df)), test_size=0.25, stratify=y, random_state=SEED)
tr, va = train_test_split(tr_all, test_size=0.20, stratify=y[tr_all], random_state=SEED)
print(f'train={len(tr)} val={len(va)} test={len(te)} | test halluc={y[te].mean()*100:.1f}%')

## 2 · Reproduce the diagnosis (deployed head + the scaler landmine)
Load the **deployed** `s1` head, score the test split (flat, AUROC≈0.6), then perturb only the near-constant feature columns by +1e-4 and watch the output collapse to a constant (AUROC → 0.5) — exactly what a new dataset does to those features.

In [ ]:
old_sc = pickle.load(open(os.path.join(ART, 'hal_det_sentence_s1_scaler.pkl'), 'rb'))
old_m = CombinedNN(32); old_m.load_state_dict(torch.load(os.path.join(ART, 'hal_det_sentence_s1_model.pth'), map_location='cpu', weights_only=True)); old_m.eval()
def score_combined(model, scaler, Xraw):
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(scaler.transform(Xraw), dtype=torch.float32))).numpy().ravel()
old_out = score_combined(old_m, old_sc, X)
print(f'DEPLOYED s1 head  test AUROC={roc_auc_score(y[te], old_out[te]):.3f}  output std={old_out.std():.3f} (flat)')
scale = np.asarray(old_sc.scale_); dead_old = np.where(scale < 1e-6)[0]
print(f'features with scale_<1e-6: {len(dead_old)}  (1/scale up to {1/scale.min():.1e})')
print('\nperturbation test on the DEPLOYED head (shift only the dead cols):')
for eps in [0.0, 1e-4, 1e-2]:
    Xp = X[te].copy(); Xp[:, dead_old] += eps
    o = score_combined(old_m, old_sc, Xp)
    print(f'  +{eps:<6}: out std={o.std():.4f} AUROC={roc_auc_score(y[te], o):.3f} max|scaled|={np.abs(old_sc.transform(Xp)).max():.1e}')

## 3 · Fix the preprocessing — neutralise the dead features
Refit `StandardScaler` on TRAIN only, then set `scale_ = 1.0` for any feature whose training variance is ~0 (`var_ < 1e-12`). Those features carry no learnable signal anyway; neutralising them means a downstream distribution shift produces a *small* value (÷1) instead of an exploding one (÷1e-13). Informative small-magnitude features are left untouched.

In [ ]:
fix_sc = StandardScaler().fit(X[tr_all])
dead_mask = fix_sc.var_ < 1e-12
fix_sc.scale_[dead_mask] = 1.0
live = np.where(~dead_mask)[0]
print(f'neutralised {int(dead_mask.sum())} constant features; {len(live)} live features kept')
Xp = X[te].copy(); Xp[:, np.where(dead_mask)[0]] += 1e-2
print(f'max|scaled| under FIXED scaler after +1e-2 shift = {np.abs(fix_sc.transform(Xp)).max():.2f} (was ~1e10)')

## 4 · Train two candidate heads on the cleaned features (TRAIN split)
**(a) `CombinedNN` trained properly** — minibatching + `BCEWithLogits(pos_weight)` + AdamW, early stop on val AUROC (vs the deployed head's single full-batch step/epoch). **(b) Regularised logistic regression** baseline. The 71-feature layout is preserved for `CombinedNN` (the fixed scaler defuses the dead cols); logreg uses the live features.

In [ ]:
torch.manual_seed(SEED); np.random.seed(SEED)
Xs = fix_sc.transform(X)
Xtr_t = torch.tensor(Xs[tr], dtype=torch.float32); ytr_t = torch.tensor(y[tr], dtype=torch.float32).unsqueeze(1)
Xva_t = torch.tensor(Xs[va], dtype=torch.float32)
pos_w = torch.tensor([(y[tr] == 0).sum() / max((y[tr] == 1).sum(), 1)], dtype=torch.float32)
net = CombinedNN(32)
opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
crit = torch.nn.BCEWithLogitsLoss(pos_weight=pos_w)
dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xtr_t, ytr_t), batch_size=32, shuffle=True)
best_auc, best_state, patience, bad = -1, None, 40, 0
for ep in range(400):
    net.train()
    for xb, yb in dl:
        opt.zero_grad(); crit(net(xb), yb).backward(); opt.step()
    net.eval()
    with torch.no_grad():
        pv = torch.sigmoid(net(Xva_t)).numpy().ravel()
    auc = roc_auc_score(y[va], pv)
    if auc > best_auc:
        best_auc, best_state, bad = auc, {k: v.detach().clone() for k, v in net.state_dict().items()}, 0
    else:
        bad += 1
        if bad >= patience:
            print(f'early stop @ epoch {ep} (best val AUROC={best_auc:.3f})'); break
net.load_state_dict(best_state); net.eval()
with torch.no_grad():
    new_nn_out = torch.sigmoid(net(torch.tensor(Xs, dtype=torch.float32))).numpy().ravel()
print(f'CombinedNN (fixed): val AUROC={best_auc:.3f}')
lr = LogisticRegression(max_iter=5000, C=1.0, class_weight='balanced')
lr.fit(Xs[tr_all][:, live], y[tr_all])
new_lr_out = lr.predict_proba(Xs[:, live])[:, 1]
print('logreg fit on', len(live), 'live features')

## 5 · Full held-out evaluation (TEST split) — old vs new candidates
AUROC / AUPR / Accuracy / Precision / Recall / F1 at the F1-optimal threshold, the confusion matrix, ROC + PR curves, and the score histogram (to confirm the new head is no longer flat).

In [ ]:
cands = {'OLD CombinedNN (s1)': old_out, f'NEW CombinedNN ({TRAIN_TAG}, fixed)': new_nn_out, f'NEW LogReg ({TRAIN_TAG})': new_lr_out}
res = {}
for name, s in cands.items():
    m = M.detector_metrics(y[te], s[te], threshold=M.best_threshold(y[te], s[te]))
    M.attach_curves(m, y[te], s[te]); res[name] = m
print(M.summary_table(res).to_string())
print('\noutput std (flat<->spread):', {k: round(float(v[te].std()), 3) for k, v in cands.items()})
for name, m in res.items():
    print(f'  {name}: CM={m["confusion_matrix"].tolist()}  [[TN,FP],[FN,TP]]')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2)); M.plot_roc(ax[0], res); M.plot_pr(ax[1], res)
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for axx, (name, m) in zip(axes, res.items()):
    M.plot_confusion(axx, m['confusion_matrix'], title=f'{name}\nAUROC={m["AUROC"]:.3f} F1={m["F1"]:.2f}')
plt.tight_layout(); plt.show()
fig, ax = plt.subplots(figsize=(8, 3.5))
for name, s in cands.items():
    ax.hist(s[te], bins=25, alpha=0.5, label=name)
ax.set_title('score distribution on TEST (flat head = single spike)'); ax.legend(fontsize=8); plt.show()

## 6 · Stability check — the landmine is gone
Re-run the perturbation on the **new** `CombinedNN` (fixed scaler). Output std should stay > 0 and AUROC stable — proof the transfer-collapse mechanism is fixed.

In [ ]:
print('perturbation test on the NEW CombinedNN (fixed scaler):')
for eps in [0.0, 1e-4, 1e-2, 1e-1]:
    Xp = X[te].copy(); Xp[:, np.where(dead_mask)[0]] += eps
    with torch.no_grad():
        o = torch.sigmoid(net(torch.tensor(fix_sc.transform(Xp), dtype=torch.float32))).numpy().ravel()
    print(f'  +{eps:<6}: out std={o.std():.4f} AUROC={roc_auc_score(y[te], o):.3f}')

## 7 · Real cross-dataset transfer eval (GPU)
Score the old vs new heads on a held-out target (`TRANSFER_DS@TRANSFER_OFFSET`). Cached to `data/claims_sent_<DS>_transfer.parquet` (squad@0 already exists from the audit), so re-runs are instant. **Leakage note:** if `TRANSFER_DS` is in `BUILD_DATASETS`, keep the offsets disjoint — then this measures held-out-question generalisation; for *pure* cross-dataset transfer choose a `TRANSFER_DS` not in the training set.

In [ ]:
if TRANSFER_DS in BUILD_DATASETS and BUILD:
    print(f'NOTE: {TRANSFER_DS} is in BUILD_DATASETS -> ensure train offset '
          f'({BUILD_OFFSETS.get(TRANSFER_DS, 0)}) and transfer offset ({TRANSFER_OFFSET}) are disjoint '
          f'(train pulls {BUILD_N} Qs). This is held-out-question generalisation, not pure transfer.')
TPATH = os.path.join(DATA, f'claims_sent_{TRANSFER_DS}_transfer.parquet')
if os.path.exists(TPATH):
    tdf = pd.read_parquet(TPATH); print('loaded cached transfer set', tdf.shape)
else:
    import retrain
    tdf, _ = retrain.gen_and_cache(TRANSFER_DS, n=TRANSFER_N, offset=TRANSFER_OFFSET, regime='sentence',
                                   label_method='llm_judge', drop_refusals=True)
    tdf.to_parquet(TPATH); print('generated + cached transfer set', tdf.shape)
yT = tdf['hallucination'].to_numpy().astype(int); XT = tdf[HS].to_numpy(np.float64)
print(f'{TRANSFER_DS} transfer: n={len(tdf)} halluc={yT.mean()*100:.1f}%')

In [ ]:
old_T = score_combined(old_m, old_sc, XT)
with torch.no_grad():
    new_T = torch.sigmoid(net(torch.tensor(fix_sc.transform(XT), dtype=torch.float32))).numpy().ravel()
new_lrT = lr.predict_proba(fix_sc.transform(XT)[:, live])[:, 1]
tcands = {'OLD CombinedNN (s1)': old_T, f'NEW CombinedNN ({TRAIN_TAG}, fixed)': new_T, f'NEW LogReg ({TRAIN_TAG})': new_lrT}
tres = {}
for name, s in tcands.items():
    m = M.detector_metrics(yT, s, threshold=M.best_threshold(yT, s)); M.attach_curves(m, yT, s); tres[name] = m
print(f'=== {TRANSFER_DS.upper()} TRANSFER (heads trained on {TRAIN_TAG}) ==='); print(M.summary_table(tres).to_string())
print('output std:', {k: round(float(v.std()), 3) for k, v in tcands.items()})
for name, m in tres.items():
    print(f'  {name}: CM={m["confusion_matrix"].tolist()}')
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for axx, (name, m) in zip(axes, tres.items()):
    M.plot_confusion(axx, m['confusion_matrix'], title=f'{name}\nAUROC={m["AUROC"]:.3f}')
plt.tight_layout(); plt.show()

## 8 · Save the new head (gated — `SAVE=False` by default)
**Do not save yet.** Review the held-out + transfer tables first. Save only if the new head beats the old on **both** (AUROC up, confusion matrix no longer degenerate). Writes `hal_det_sentence_<TAG>_*` — never overwrites the deployed `s1`. The kept-feature index list is saved so inference applies the same neutralised scaler.

In [ ]:
SAVE = False               # <-- leave False for now (per instruction). Flip only after the tables look good.
WHICH = 'combinednn'       # 'combinednn' or 'logreg'
if SAVE:
    tag = TRAIN_TAG
    pickle.dump(fix_sc, open(os.path.join(ART, f'hal_det_sentence_{tag}_scaler.pkl'), 'wb'))
    np.save(os.path.join(ART, f'hal_det_sentence_{tag}_live_features.npy'), live)
    if WHICH == 'combinednn':
        torch.save(net.state_dict(), os.path.join(ART, f'hal_det_sentence_{tag}_model.pth'))
    else:
        pickle.dump(lr, open(os.path.join(ART, f'hal_det_sentence_{tag}_logreg.pkl'), 'wb'))
    print('saved', WHICH, f'-> hal_det_sentence_{tag}_*  (s1 artifacts untouched)')
else:
    print('SAVE=False — nothing written. Review the eval + transfer tables, then set SAVE=True.')